# NovaBank Digital Graph Investigation

This notebook introduces relationship and network analysis for the **NovaBank Digital** digital-banking track. A learner builds a graph from the canonical generated tables, computes explainable network features, and investigates digital-banking network structures: mule rings, shared devices, shared beneficiaries, and pass-through clusters.

Graph evidence **extends** the existing v0.4 digital-banking tabular investigation — it does not replace it. Every network signal is connected back to the session-risk, beneficiary-novelty, pass-through, and shared-device tabular signals and the v0.5 case context.

The **User** (digital login identity) is distinct from the **Client** (legal customer): a Client owns the relationship, a User authenticates sessions. Both are nodes here, and the distinction matters when reading shared-device signals.

In [ ]:
import pandas as pd

from banking_fraud_lab import (
    build_all_graph_features,
    build_banking_graph,
    build_foundation_progressive_views,
    generate_minimal_banking_world,
    join_graph_features_to_view,
)

pd.set_option('display.max_columns', None)

## Build the Banking Graph

The graph is derived purely from the canonical generated tables. Sessions and devices become nodes: a session authenticates a **User**, and a device is linked to every session observed on its fingerprint. Several **Users** sharing one device is the core shared-device signal.

In [ ]:
tables = generate_minimal_banking_world(seed=42, scale='tiny')
graph = build_banking_graph(tables)

print(f'Nodes: {graph.number_of_nodes()}')
print(f'Edges: {graph.number_of_edges()}')

node_type_counts = (
    pd.Series([d['node_type'] for _, d in graph.nodes(data=True)])
    .value_counts()
    .sort_index()
)
node_type_counts

## Mule Rings and Connected Components

A mule ring is a connected cluster of accounts, beneficiaries, and **Users** that move funds between each other. Connected-component membership sizes the clusters; large components spanning many **Users** or beneficiaries are the network signature of a ring.

In [ ]:
features = build_all_graph_features(graph)
components = features['graph_connected_component']
ring_candidates = (
    components[components['component_size'] > 1]
    .sort_values('component_size', ascending=False)
)
ring_candidates.head(10)

## Shared Devices

Devices are derived from session telemetry: each distinct `device_fingerprint_hash` is one device node, and each session links its device to its **User**. A device with high degree (many **Users**) is a shared-device lead — the kind of signal that, combined with pass-through velocity, points at account takeover or mule coordination.

In [ ]:
degree = features['graph_node_degree']
device_hubs = (
    degree[degree['node_type'] == 'device']
    .sort_values('node_degree', ascending=False)
    .head(5)
)
device_hubs

## Shared Beneficiaries and Pass-Through

Beneficiaries that receive funds from several otherwise-unrelated **Clients** or **Users** are a counterparty-network signal. Bridge nodes highlight single points through which funds pass between clusters — the structural core of a pass-through cluster.

In [ ]:
bridge = features['graph_bridge_node']
beneficiary_bridges = bridge[
    (bridge['node_type'] == 'beneficiary')
    & (bridge['is_bridge_endpoint'] == 1)
]
print(f'Beneficiary bridge nodes: {len(beneficiary_bridges)}')
beneficiary_bridges.head(10)

## Connect Graph Evidence to the Investigation

Graph features are joined back to the foundation `foundation_alert_lifecycle` Progressive data view so network evidence supports the alert-level investigation. The join does not drop or duplicate any keyed row — it adds network context to the existing tabular alert lifecycle.

In [ ]:
views = build_foundation_progressive_views(tables)
alert_view = views['foundation_alert_lifecycle']
augmented = join_graph_features_to_view(degree, alert_view)
network_columns = [c for c in augmented.columns if c.startswith('node_degree_')]
view_keys = ['alert_id', 'user_id', 'session_id']
augmented[[c for c in view_keys if c in augmented.columns] + network_columns].head()

## Interpretation and Limitations

Network signals here are **investigative leads**, not proof of fraud. A large component, a high-degree device, or a bridge beneficiary is a reason to look more closely at the v0.4 digital-banking tabular signals (session risk, beneficiary novelty, pass-through, shared device) and the v0.5 case context — not a verdict. This notebook avoids headline accuracy claims and frames outputs for business, risk, and compliance discussion.

- **Mule rings** (large components) are structural; confirm with beneficiary-change history and account age.
- **Shared devices** are leads; confirm with session telemetry and auth method.
- **Shared beneficiaries / pass-through** are leads; confirm with payment velocity and purpose.

Remember the **User** (login identity) vs **Client** (legal customer) distinction when reading shared-device and counterparty signals.

All data is synthetic (NovaBank Digital). No real client data, no reconstruction of real events, and no legal advice is provided.